In [ ]:
import os
import copy
import time
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

from sklearn.metrics import classification_report, confusion_matrix

랜덤 시드 및 CPU 설정

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")
print("사용 장치:", device)

데이터 경로

In [ ]:
DATA_DIR = Path(r"C:\Brain MRI_Alzheimer")

# 기본 경로
train_dir = DATA_DIR / "train"
val_dir = DATA_DIR / "val"
test_dir = DATA_DIR / "test"

# 만약 train/train, val/val, test/test 구조라면 자동 수정
if (train_dir / "train").exists():
    train_dir = train_dir / "train"

if (val_dir / "val").exists():
    val_dir = val_dir / "val"

if (test_dir / "test").exists():
    test_dir = test_dir / "test"

print("train_dir:", train_dir)
print("val_dir:", val_dir)
print("test_dir:", test_dir)

print("train 존재:", train_dir.exists())
print("val 존재:", val_dir.exists())
print("test 존재:", test_dir.exists())

print("\ntrain 내부 폴더:")
for p in train_dir.iterdir():
    print("-", p.name)

Resnet은 입력이 3채널로 이미지 전처리

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

val_test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

Dataset/Dateloader 생성

In [ ]:
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=val_test_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=val_test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

class_names = train_dataset.classes
num_classes = len(class_names)

print("클래스:", class_names)
print("클래스 개수:", num_classes)
print("Train 개수:", len(train_dataset))
print("Val 개수:", len(val_dataset))
print("Test 개수:", len(test_dataset))

print("\nTrain 클래스별 개수:", np.bincount(train_dataset.targets))
print("Val 클래스별 개수:", np.bincount(val_dataset.targets))
print("Test 클래스별 개수:", np.bincount(test_dataset.targets))

데이터중복 확인

In [ ]:
train_files = set([Path(p).name for p, _ in train_dataset.samples])
val_files = set([Path(p).name for p, _ in val_dataset.samples])
test_files = set([Path(p).name for p, _ in test_dataset.samples])

print("train-val 중복 파일명 개수:", len(train_files & val_files))
print("train-test 중복 파일명 개수:", len(train_files & test_files))
print("val-test 중복 파일명 개수:", len(val_files & test_files))

클래스 불균형 보정

In [ ]:
targets = train_dataset.targets
class_counts = np.bincount(targets)

class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * num_classes
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("클래스별 이미지 수:", class_counts)
print("class weights:", class_weights)

ResNet18 모델 정의

In [ ]:
model = models.resnet18(weights=None)

# ResNet18 마지막 fully connected layer 수정
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

model = model.to(device)

print(model)

loss/optimizer 설정

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

학습 함수 / 평가 함수

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            running_loss += loss.item() * images.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc, all_labels, all_preds, all_probs

모델 학습

In [ ]:
EPOCHS = 15

best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

train_losses = []
val_losses = []
train_accs = []
val_accs = []

start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    val_loss, val_acc, _, _, _ = evaluate(
        model, val_loader, criterion, device
    )
    
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc*100:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "best_resnet18_alzheimer.pth")
        print("Best model saved!")

elapsed = time.time() - start_time

print("\n학습 완료")
print(f"소요 시간: {elapsed/60:.2f}분")
print(f"Best Val Acc: {best_val_acc*100:.2f}%")

model.load_state_dict(best_model_wts)

학습 그래프 출력

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train / Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot([acc * 100 for acc in train_accs], label="Train Accuracy")
plt.plot([acc * 100 for acc in val_accs], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Train / Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

Test 데이터 평가

In [ ]:
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
))

confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45, ha="right")
plt.yticks(tick_marks, class_names)

for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(
            j, i, cm[i, j],
            horizontalalignment="center",
            verticalalignment="center"
        )

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

confusion matrix 비율

In [ ]:
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(7, 6))
plt.imshow(cm_norm, interpolation="nearest")
plt.title("Normalized Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45, ha="right")
plt.yticks(tick_marks, class_names)

for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(
            j, i, f"{cm_norm[i, j]*100:.1f}%",
            horizontalalignment="center",
            verticalalignment="center"
        )

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

Grad-CAM 함수 정의

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        
        self.gradients = None
        self.activations = None
        
        self.forward_hook = self.target_layer.register_forward_hook(self.save_activation)
        self.backward_hook = self.target_layer.register_full_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        
        output = self.model(input_tensor)
        
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        
        self.model.zero_grad()
        
        score = output[:, target_class]
        score.backward()
        
        gradients = self.gradients
        activations = self.activations
        
        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)
        
        cam = torch.relu(cam)
        
        cam = torch.nn.functional.interpolate(
            cam,
            size=input_tensor.shape[2:],
            mode="bilinear",
            align_corners=False
        )
        
        cam = cam.squeeze().cpu().numpy()
        
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        
        return cam, output
    
    def remove_hooks(self):
        self.forward_hook.remove()
        self.backward_hook.remove()

Grad-CAM 시각화 함수

In [ ]:
def denormalize_image(tensor_img):
    """
    Normalize(mean=0.5, std=0.5)를 되돌림.
    입력: C x H x W tensor
    출력: H x W x C numpy image
    """
    img = tensor_img.clone().cpu()
    img = img * 0.5 + 0.5
    img = img.numpy()
    img = np.transpose(img, (1, 2, 0))
    img = np.clip(img, 0, 1)
    return img


def show_gradcam(model, image_path, transform, class_names, device):
    image = Image.open(image_path).convert("L")
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    target_layer = model.layer4[-1]
    gradcam = GradCAM(model, target_layer)
    
    cam, output = gradcam.generate(input_tensor)
    gradcam.remove_hooks()
    
    probs = torch.softmax(output, dim=1)
    pred_idx = output.argmax(dim=1).item()
    confidence = probs[0, pred_idx].item()
    
    original_img = denormalize_image(input_tensor.squeeze(0))
    
    heatmap = plt.cm.jet(cam)[:, :, :3]
    overlay = 0.45 * heatmap + 0.55 * original_img
    overlay = np.clip(overlay, 0, 1)
    
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 3, 1)
    plt.imshow(original_img, cmap="gray")
    plt.title("Original MRI")
    plt.axis("off")
    
    plt.subplot(1, 3, 2)
    plt.imshow(cam, cmap="jet")
    plt.title("Grad-CAM Heatmap")
    plt.axis("off")
    
    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title(f"Pred: {class_names[pred_idx]}\nConf: {confidence*100:.2f}%")
    plt.axis("off")
    
    plt.tight_layout()
    plt.show()
    
    print("이미지 경로:", image_path)
    print("예측 클래스:", class_names[pred_idx])
    print(f"신뢰도: {confidence*100:.2f}%")

Test 이미지 하나로 Grad-CAM 확인

In [ ]:
sample_path, sample_label = test_dataset.samples[0]

print("정답 클래스:", class_names[sample_label])
print("이미지 경로:", sample_path)

show_gradcam(
    model=model,
    image_path=sample_path,
    transform=val_test_transform,
    class_names=class_names,
    device=device
)

클래스별 Grad-CAM 확인

In [ ]:
def get_one_sample_per_class(dataset, class_names):
    selected = {}
    
    for path, label in dataset.samples:
        class_name = class_names[label]
        if class_name not in selected:
            selected[class_name] = (path, label)
        
        if len(selected) == len(class_names):
            break
    
    return selected


samples_per_class = get_one_sample_per_class(test_dataset, class_names)

for class_name, (path, label) in samples_per_class.items():
    print("=" * 60)
    print("정답 클래스:", class_name)
    
    show_gradcam(
        model=model,
        image_path=path,
        transform=val_test_transform,
        class_names=class_names,
        device=device
    )

맞춘 이미지와 틀린 이미지 Grad-CAM 비교

In [ ]:
def find_correct_and_wrong_samples(model, dataset, transform, class_names, device, max_samples=3):
    model.eval()
    
    correct_samples = []
    wrong_samples = []
    
    for path, true_label in dataset.samples:
        image = Image.open(path).convert("L")
        input_tensor = transform(image).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(input_tensor)
            pred_label = output.argmax(dim=1).item()
        
        if pred_label == true_label and len(correct_samples) < max_samples:
            correct_samples.append((path, true_label, pred_label))
        
        if pred_label != true_label and len(wrong_samples) < max_samples:
            wrong_samples.append((path, true_label, pred_label))
        
        if len(correct_samples) >= max_samples and len(wrong_samples) >= max_samples:
            break
    
    return correct_samples, wrong_samples


correct_samples, wrong_samples = find_correct_and_wrong_samples(
    model=model,
    dataset=test_dataset,
    transform=val_test_transform,
    class_names=class_names,
    device=device,
    max_samples=3
)

print("맞춘 샘플 수:", len(correct_samples))
print("틀린 샘플 수:", len(wrong_samples))